In [ ]:
!nvidia-smi

Thu Nov 19 09:43:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install roboflow

from roboflow import Roboflow
import os
# Set ROBOFLOW_API_KEY in Colab Secrets or environment before running
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY"))
project = rf.workspace("sjp-gcfju").project("2026_greg")
version = project.version(4)
dataset = version.download("yolov11")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 114.3 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 2026_GreG-4 in yolov11:: 100%|██████████| 3964/3964 [00:00<00:00, 6713.37it/s]


In [ ]:
from pathlib import Path
import yaml

dataset_root = Path(dataset.location).resolve()
original_yaml = dataset_root / "data.yaml"

config = yaml.safe_load(original_yaml.read_text(encoding="utf-8"))

print("Dataset root: ", dataset_root)
print("Classes: ", config["names"])
print("Train: ", config.get("train"))
print("Val: ", config.get("val"))
print("Test: ", config.get("test"))


Dataset root:  /content/2026_GreG-4
Classes:  ['Back', 'Front', 'left_side', 'right_side', 'upright', 'upside_down']
Train:  ../train/images
Val:  ../valid/images
Test:  ../test/images


In [ ]:
from pathlib import Path
import yaml

dataset_root = Path(dataset.location).resolve()
config = yaml.safe_load(
    (dataset_root / "data.yaml").read_text(encoding="utf-8")
)

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ["train", "valid", "test"]:
    image_dir = dataset_root / split / "images"
    label_dir = dataset_root / split / "labels"

    assert image_dir.is_dir(), f"Image directory not found: {image_dir}"
    assert label_dir.is_dir(), f"Label directory not found: {label_dir}"

    count = sum(
        p.is_file() and p.suffix.lower() in extensions
        for p in image_dir.iterdir()
    )
    print(f"{split}: {count} images")
    assert count > 0, f"{split} has no images"

config["path"] = str(dataset_root)
config["train"] = str(dataset_root / "train/images")
config["val"] = str(dataset_root / "valid/images")
config["test"] = str(dataset_root / "test/images")

data_yaml = dataset_root / "data_colab.yaml"
data_yaml.write_text(
    yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("Classes: ", config["names"])
print("Training config ready: ", data_yaml)


train: 1932 images
valid: 26 images
test: 22 images
Classes:  ['Back', 'Front', 'left_side', 'right_side', 'upright', 'upside_down']
Training config ready:  /content/2026_GreG-4/data_colab.yaml


In [ ]:

%pip install -q ultralytics==8.4.215

import torch
import ultralytics

print("Ultralytics version: ", ultralytics.__version__)
assert torch.cuda.is_available(), "GPU not connected; please check the runtime"
print("GPU: ", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics version:  8.4.215
GPU:  Tesla T4


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
from ultralytics import YOLO
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import torch

assert torch.cuda.is_available(), "GPU not connected"

# Use the exact same dataset as the previous training run
data_yaml = Path("/content/2026_GreG-4/data_colab.yaml")
assert data_yaml.is_file(), "Data config not found; check whether the dataset is still present in Colab"
assert Path("/content/drive/MyDrive").is_dir(), "Please mount Google Drive first"

# New run directory name
run_name = "paper_cups_416_full120_" + datetime.now(
    ZoneInfo("Europe/Stockholm")
).strftime("%Y%m%d_%H%M%S")

print("Training data config: ", data_yaml)
print("Run name: ", run_name)

# Use the exact same initial weights as the previous run
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(data_yaml),
    imgsz=416,
    epochs=120,
    batch=8,
    device=0,
    workers=2,
    patience=0,       # Disable early stopping
    seed=42,
    optimizer="auto",

    fliplr=0.0,
    flipud=0.0,
    degrees=0.0,
    mosaic=0.0,
    mixup=0.0,

    project="/content/drive/MyDrive/paper_cup_yolo11",
    name=run_name,
    exist_ok=False,
    plots=True,
    save=True,
)

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"

print("Run directory: ", run_dir)
print("Best weights: ", best_pt)


Training data config:  /content/2026_GreG-4/data_colab.yaml
Run name:  paper_cups_416_full120_20261119_114137
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/2026_GreG-4/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import yaml

run_dir = Path(
    "/content/drive/MyDrive/paper_cup_yolo11/"
    "paper_cups_416_20261119_110233"
)

# Read actual training arguments to confirm batch size and data path
args = yaml.safe_load(
    (run_dir / "args.yaml").read_text(encoding="utf-8")
)

for key in ["data", "imgsz", "epochs", "batch", "patience", "optimizer"]:
    print(f"{key}: {args.get(key)}")

data_path = Path(args["data"])
assert data_path.is_file(), f"Data config not found: {data_path}"

model = YOLO(str(run_dir / "weights" / "best.pt"))

metrics = model.val(
    data=str(data_path),
    split="test",
    imgsz=416,
    batch=8,
    device=0,
    workers=2,
    plots=True,
    project=str(run_dir),
    name="test_evaluation",
)

print("Test mAP50: ", metrics.box.map50)
print("Test mAP50-95: ", metrics.box.map)
print("Evaluation output directory: ", metrics.save_dir)


data: /content/2026_GreG-4/data_colab.yaml
imgsz: 416
epochs: 120
batch: 8
patience: 20
optimizer: auto
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,266 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1222.4±326.7 MB/s, size: 47.1 KB)
val: Scanning /content/2026_GreG-4/test/labels... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 570.7it/s 0.0s
val: New cache created: /content/2026_GreG-4/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 4.2it/s 0.7s
                   all         22         63      0.908      0.919      0.951      0.788
                  Back          4          9      0.986      0.889      0.975      0.817
                 Front          4          5      0.979          1      0.995      0.847
             left_side          9         15      0.923      0.8

In [ ]:
from pathlib import Path
from ultralytics import YOLO

run_dir = Path(
    "/content/drive/MyDrive/paper_cup_yolo11/"
    "paper_cups_416_full120_20261119_114137"
)

model = YOLO(str(run_dir / "weights" / "best.pt"))

metrics = model.val(
    data="/content/2026_GreG-4/data_colab.yaml",
    split="test",
    imgsz=416,
    batch=8,
    device=0,
    workers=2,
    plots=True,
    project=str(run_dir),
    name="test_evaluation",
)

print("Test mAP50: ", metrics.box.map50)
print("Test mAP50-95: ", metrics.box.map)
print("Evaluation output directory: ", metrics.save_dir)


Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,266 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1345.6±439.8 MB/s, size: 47.1 KB)
val: Scanning /content/2026_GreG-4/test/labels.cache... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 6.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.4it/s 1.2s
                   all         22         63      0.954      0.939      0.957      0.796
                  Back          4          9      0.988      0.889      0.885      0.695
                 Front          4          5       0.97          1      0.995      0.891
             left_side          9         15      0.934      0.938      0.937      0.763
            right_side          3          4       0.84          1      0.995       0.82
               upright          9         17    

In [1]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/paper_cup_yolo11/"
    "paper_cups_416_full120_20261119_114137/weights/best.pt"
)

FileNotFoundError: Cannot find file: /content/drive/MyDrive/paper_cup_yolo11/paper_cups_416_full120_20261119_114137/weights/best.pt